In [1]:
import sys 
import os 

sys.path.append(os.path.join('..', '..'))

In [2]:
import pandas as pd 
import unicodedata
import re
from pathlib import Path

from config.data_cleaning_config import (
    DEFAULT_DROP_COLUMNS,
    COLUNAS_TEXTO,
    COLUNAS_BINARIAS,
    COLUNA_DIA_SEMANA,
    COLUNA_HORA,
    COLUNA_BAIRRO,
    COLUNA_NATUREZA_DESCRICAO,
)

from config.bairro_cleaning_config import (
    VALORES_INVALIDOS_BAIRRO,
    REGIAO_METROPOLITANA,
    MAPA_BAIRRO,
    BAIRROS_OFICIAIS
)

from config.natureza_descricao_cleaning_config import (
    CORRECOES,
    CRIME_VIOLENTO,
    ATENDIMENTO_OPERACIONAL_ASSISTENCIAL,
    ACIDENTE_TRANSITO,
    ACIDENTE_NATURAL,
    CRIME_PATRIMONIAL,
    CRIME_ADMNISTRACAO_PUBLICA,
    CRIME_HONRA_DISCRIMINACAO,
    CRIME_CRIANCA_ADOLESCENTE,
    CRIME_FRAUDE_DOCUMENTAL,
    CRIME_DROGAS_SUBSTANCIAS,
    CRIME_ORDEM_PUBLICA,
    RISCO_ESTRUTURAL,
    EXPLOSIVOS_E_PRODUTOS_PERIGOSOS,
    PESSOAS_DESAPARECIDAS,
    MATERIAIS_OBJETOS,
    INDETERMINADO
)

In [3]:
data_dir = Path("../../data/sigesguarda/base_de_dados")
arquivo = data_dir / "base_unificada.csv"

In [4]:
output_dir = Path("../../data/sigesguarda/cleaned")
output_dir.mkdir(parents=True, exist_ok=True)

arquivo_limpo = output_dir / "base_unificada.csv"

In [5]:
df = pd.read_csv(arquivo, sep=";", encoding="latin1", dtype=str)

## 01. Padronização dos dados de texto

Precisamos de uma função que faça a limpeza dos campos de texto da base de dados com o objetivo de padronizar os valores. Por exemplo, 'Centro Cívico' e 'centro civico' devem ser tratados como iguais. Para isso, iremos remover a acentuação, converter tudo para letras minúsculas e eliminar caracteres especiais, mantendo apenas letras, números e espaços. 

In [6]:
def limpar_texto(valor):
    if pd.isna(valor):
        return pd.NA
    
    valor = str(valor).strip().lower()
    
    valor = unicodedata.normalize("NFKD", valor)
    valor = valor.encode("ascii", "ignore").decode("utf-8")
    
    valor = re.sub(r"[^a-z0-9\s]", " ", valor)
    valor = re.sub(r"\s+", " ", valor).strip()
    
    return valor if valor else pd.NA


Essa função irá executar:

1. Tratamento de valores nlos: se o valor for NaN, a função retorna `pd.NA`
2. Conversão para string e limpeza básica: o valor é convertido para string, removeremos espaços extras no início e no fim (`.strip()`) e transformamos tudo em letras minúsculas (.lower())
3. Remoção de acentos: usamos `unicodedata.normalize("NFKD", valor)` para decompor os caracteres acentuados (ex: "ção" vira "c~a~o"), e depois `.encode("ascii", "ignore")` elimina os acentos, mantendo apenas os caracteres ASCII básicos.
4. Filtragem de caracteres: com `re.sub(r"[^a-z0-9\s]", " ", valor)`, substituímos qualquer caractere que não seja letra minúscula, número ou espaço por um espaço em branco. Isso remove pontuação, símbolos e outros caracteres especiais. Esse espaço em branco é normalizado com `re.sub(r"\s+", " ", valor).strip()`, onde é substituida sequência de espaços por um único espaço e `.strip()` remove os espaços desnecessários das bordas. 

Ao fim, se toda a limpeza retornar uma string vazia, retornamos pd.NA. Caso contrário, retornamos o texto limpo.

## 02. Padronização de colunas binárias

Para padronizar colunas binárias, definindo `0` como `False` e `1` como `True`, é necessário primeiro identificar todos os valores distintos presentes na base de dados. Isso garante que o mapeamento seja feito corretamente. Para identificar todos os valores únicos na tupla `COLUNAS_BINARIAS`, disponibilizado em `config/data_cleaning_config.py`, faremos:

In [7]:
valores_unicos = {col: set() for col in COLUNAS_BINARIAS}

In [8]:
for col in COLUNAS_BINARIAS:
    if col in df.columns:
        valores = df[col].dropna().unique()
        valores_unicos[col].update(valores)

for col, valores in valores_unicos.items():
    print(f"\n{col}:")
    for valor in sorted(valores):
        print(valor)


FLAG_EQUIPAMENTO_URBANO:
NÃO
SIM
f
t

FLAG_FLAGRANTE:
NÃO
NÃO
SIM

NATUREZA1_DEFESA_CIVIL:
0
1

NATUREZA2_DEFESA_CIVIL:
0
1

NATUREZA3_DEFESA_CIVIL:
0
1

NATUREZA4_DEFESA_CIVIL:
0
1

NATUREZA5_DEFESA_CIVIL:
0


In [9]:
def mapear_binario(valor):

    if pd.isna(valor):
        return 0
    
    s = str(valor).strip().lower()

    if s in ('1.0', '0.0'):
        return 1 if s == '1.0' else 0

    valor_limpo = limpar_texto(valor)

    if pd.isna(valor_limpo):
        return 0

    if valor_limpo in ('sim', 'y', 't', '1'):
        return  1
    
    if valor_limpo in ('n', 'nao', 'f', '0', '-----------------------',
                        '--------------', '----------------------',
                        '----------------------', '----------------------', 
                        '----------------------', '----------------------'):
        return 0
    
    return 0

## 03. Mapeamento dos dias da semana

Assim como foi feito para as colunas binárias, iremos encontrar os valores únicos para a coluna `OCORRENCIA_DIA_SEMANA` e fazer o mapeamento para transformar em inteiro.

In [10]:
valores_unicos = {col: set() for col in COLUNA_DIA_SEMANA}

In [11]:
for col in COLUNA_DIA_SEMANA:
    if col in df.columns:
        valores = df[col].dropna().unique()
        valores_unicos[col].update(valores)

for col, valores in valores_unicos.items():
    print(f"\n{col}:")
    for valor in sorted(valores):
        print(valor)


OCORRENCIA_DIA_SEMANA:
DOMINGO
Domingo
QUARTA
QUINTA
Quarta
Quinta
SEGUNDA
SEXTA
Segunda
Sexta
SÁBADO
SÃ¡bado
TERÇA
TerÃ§a


In [12]:
def mapear_dia_semana(valor):

    if pd.isna(valor):
        return 0
    
    texto_limpo = limpar_texto(valor)

    if pd.isna(texto_limpo) or texto_limpo == '':
        return 0
    
    mapa_dias = {
        'domingo': 1,
        'segunda': 2,
        'terca': 3,
        'quarta': 4,
        'quinta': 5,
        'sexta': 6,
        'sabado': 7
    }

    return mapa_dias.get(texto_limpo, 0)

## 04. Tratamento da coluna OCORRENCIA_HORA

Iremos padronizar a coluna OCORRENCIA_HORA que possui os registros no formato HH:MM:SS (hora, minuto e segundo) e, a partir dela, criar as colunas adicionais hora e minuto, descartando os segundos.

In [13]:
def tratar_coluna_hora(df, coluna):

    if coluna not in df.columns:
        return df 
    
    serie_hora = pd.to_datetime(df[coluna], format="%H:%M:%S", errors="coerce")

    df[coluna] = serie_hora.dt.strftime("%H:%M:%S")

    df[f"{coluna}_HORA"] = serie_hora.dt.hour.astype("Int64")
    df[f"{coluna}_MINUTO"] = serie_hora.dt.minute.astype("Int64")

    df.drop(columns=[coluna], inplace=True)
    
    return df 

## 05. Flags: MANHA, TARDE, NOITE, MADRUGADA

Vamos criar quatro colunas binária, indicando em qual período do dia a hora de uma ocorrência se enquadrada: madrugada, manhã, tarde ou noite. Cada coluna recebe o valor `1` (verdadeiro) ou `0` (falso). Para isso, utilizaremos os intervalos:

- Madrugada: 00:00 - 05:59
- Manhã: 06:00 - 11:59
- Tarde: 12:00 - 17:59
- Noite: 18:00 - 23:59

In [14]:
def adicionar_periodo_dia(df, coluna_hora):

    if coluna_hora not in df.columns:
        return df 
    
    horas = pd.to_numeric(df[coluna_hora], errors="coerce")

    df["MADRUGADA"] = ((horas >= 0) & (horas <= 5)).astype("Int64")
    df["MANHA"] = ((horas >= 6) & (horas <= 11)).astype("Int64")
    df["TARDE"] = ((horas >= 12) & (horas <= 17)).astype("Int64")
    df["NOITE"] = ((horas >= 18) & (horas <= 23)).astype("Int64")

    return df

## 06. Erros de digitação e padronizações

Precisaremos investigar erros de digitação, abreviações e similaridades nas seguintes colunas:

- `ATENDIMENTO_BAIRRO_NOME`;
- `NATUREZA1_DESCRICAO`;
- `NATUREZA2_DESCRICAO`;
- `NATUREZA3_DESCRICAO`;
- `NATUREZA4_DESCRICAO`;
- `NATUREZA5_DESCRICAO`;

O tratamento consistirá em analisar os valores únicos de cada coluna e desenvolver scripts específicos para cada caso.

#### `ATENDIMENTO_BAIRRO_NOME`

Vamos investigar valores únicos no cadastro dos nomes dos bairros:

In [15]:
valores_unicos = {col: set() for col in COLUNA_BAIRRO}

for col in COLUNA_BAIRRO:
    if col in df:
        valores = df[col].apply(limpar_texto).dropna().unique()
        valores_unicos[col].update(valores)

for col, valores in valores_unicos.items():
    print(f"\n{col}:")
    for valor in sorted(valores):
        print(valor)


ATENDIMENTO_BAIRRO_NOME:
abranches
afonso pena
agua verde
aguas belas
ahao
ahu
alto boqueirao
alto da gla3ria
alto da gloria
alto da rua xv
atuba
augusta
bacacheri
bairro alto
bairro ficticio
bairro nao informado
bairro nao localizad
bairro oliveira
barigua
barigui
barreirinha
barro preto
batel
belas aguas
bigorrilho
boa vista
bom retiro
boqueirao
borda do campo
braga
butiatuvinha
cabral
cachoeira
cajuru
campina do siqueira
campinha grande do s
campo comprido
campo de santana
campo de sao benedit
campo largo
campo pequeno
canguiri
capanema
capao da imbuia
capao raso
casa
cascatinha
caximba
centro
centro cavico
centro civico
champagnat
cic
cidade industrial
cidade industrial de
cidade jardim
colombo
colonia faria
colonia sao venancio
cristo rei
ecoville
estados
estancia pinhais
eucaliptos
fanny
fazendinha
ferraria
formoso
francisco gorski
ganchinho
gralha azul
guaara
guabirotuba
guaira
guatupe
hauer
higiena3polis
hugo lange
iguacu 01
iguacu 1
indicacoes cancelada
ipe 2
jardim bela vist

Há diversos casos de erros de digitação, similaridades nos dados e exemplos que precisam ser excluidos, por exemplo:

- `centro cavico` (erro de digitação para `centro civico`);
- `alto da gla3ria` (erro de digitação para `alto da gloria`)
- `cic` (abreviação de `cidade industrial`);
- `bairro ficticio` (dado que precisa ser excluído).

Para evitar perda de informação e facilitar a análise, criaremos um mapeamento na pasta config. Esse mapa será responsável por aplicar os ajutes necessários e padronizar todos os valores de entrada existentes.

Na pasta `config`, foi criado o arquivo `bairro_cleaning_config.py`, que contém três mapas:

- `VALORES_INVALIDOS_BAIRRO`;
- `REGIAO_METROPOLITANA`;
- `MAPA_BAIRRO`.

O primeiro mapa armazena valores de entrada que serão descartados por não fornecerem informação precisa sobre a localização, o que prejudicaria a análise dos dados. O segundo também será descartado, pois se refere à região metropolitana de Curitiba e não ao município em si. Já o terceiro contém correções para erros de digitação em nomes de bairros que pertencem a Curitiba.

In [16]:
COLUNA_BAIRRO_NOME = COLUNA_BAIRRO[0]
BAIRROS_DESCARTAR = VALORES_INVALIDOS_BAIRRO | REGIAO_METROPOLITANA

In [17]:
def padronizar_bairro(valor):
    bairro = limpar_texto(valor)

    if pd.isna(bairro):
        return pd.NA 
    
    if bairro in BAIRROS_DESCARTAR:
        return pd.NA 
    
    bairro_padronizado = MAPA_BAIRRO.get(bairro, bairro)

    if bairro_padronizado not in BAIRROS_OFICIAIS:
        return pd.NA
    
    return bairro_padronizado

In [18]:
if COLUNA_BAIRRO_NOME in df.columns:
    df[COLUNA_BAIRRO_NOME] = df[COLUNA_BAIRRO_NOME].apply(padronizar_bairro)

In [19]:
df['ATENDIMENTO_BAIRRO_NOME'].unique()

<StringArray>
[  'cidade industrial',          'fazendinha',             'uberaba',
       'sitio cercado',           'tatuquara',       'santa candida',
           'boqueirao',              'centro',           'boa vista',
              'taboao',               'xaxim',          'pilarzinho',
            'reboucas',          'agua verde',               'batel',
          'novo mundo',      'alto boqueirao',          'capao raso',
     'jardim botanico',              'portao',             'orleans',
    'santa felicidade',          'cascatinha',     'capao da imbuia',
         'barreirinha',           'seminario',      'campo comprido',
         'prado velho',         'pinheirinho',        'butiatuvinha',
 'campina do siqueira',              'cajuru',       'sao francisco',
       'centro civico',            'sao braz',              'umbara',
             'caximba',       'jardim social',           'bacacheri',
    'campo do santana',        'santo inacio', 'jardim das americas',
      

#### `NATUREZA_DESCRICAO`

Investigaremos os valores únicos presentes nas seguintes colunas do cadastro:

- NATUREZA1_DESCRICAO;
- NATUREZA2_DESCRICAO;
- NATUREZA3_DESCRICAO;
- NATUREZA4_DESCRICAO;
- NATUREZA5_DESCRICAO;

Para isso, exportaremos os valores únicos de cada colunas para um arquivo `.txt`, que servirá de base para análise individualizada.

In [20]:
valores_unicos = set()

for col in COLUNA_NATUREZA_DESCRICAO:
    if col in df.columns:
        valores = df[col].apply(limpar_texto).dropna()
        valores_unicos.update(valores)

with open(data_dir / "valores_unicos_natureza_descricao.txt", "w", encoding="utf-8") as arquivo_valores_unicos:
    for valor in sorted(valores_unicos):
        arquivo_valores_unicos.write(f"{valor}\n")

Assim como com a coluna de bairros, também há erros de digitração e casos similares nas colunas de natureza das ocorrências. Por exemplo:

- `ameaaa` (deveria ser `ameaca`)
- `perseguiaao` (deveria ser `perseguicao`)

Além da correção desses erros, criaremos features que classificam o tipo de ocorrência registrada. Serão features boolenas, indicando como a ocorrência se relacioada com cada uma das subcategorias:

- Crime violento;
- Atendimento operacional assistencial;
- Acidente de trânsito;
- Acidente natural;
- Crime patrimonial;
- Crime de administração pública;
- Crime quanto a honra e discriminação;
- Crime criança adolescente;
- Crime fraude documental;
- Crime drogas e substancias licitas e ilicitas;
- Crime ordem publica;
- Risco estrutural;
- Explosivos e produtos perigosos; 
- Pessoas desaparecidas;
- Materiais e objetos;

O objetivo dessas features é transformar os dados textuais em dados estruturados, facilitando análises quantitativas e estatísticas.

In [21]:
MAPAS_NATUREZA_DESCRICAO = {
    "CRIME_VIOLENTO": CRIME_VIOLENTO,
    "ATENDIMENTO_OPERACIONAL_ASSISTENCIAL": ATENDIMENTO_OPERACIONAL_ASSISTENCIAL,
    "ACIDENTE_TRANSITO": ACIDENTE_TRANSITO,
    "ACIDENTE_NATURAL": ACIDENTE_NATURAL,
    "CRIME_PATRIMONIAL": CRIME_PATRIMONIAL,
    "CRIME_ADMINISTRACAO_PUBLICA": CRIME_ADMNISTRACAO_PUBLICA,
    "CRIME_HONRA_DISCRIMINACAO": CRIME_HONRA_DISCRIMINACAO,
    "CRIME_CRIANCA_ADOLESCENTE": CRIME_CRIANCA_ADOLESCENTE,
    "CRIME_FRAUDE_DOCUMENTAL": CRIME_FRAUDE_DOCUMENTAL,
    "CRIME_DROGAS_SUBSTANCIAS": CRIME_DROGAS_SUBSTANCIAS,
    "CRIME_ORDEM_PUBLICA": CRIME_ORDEM_PUBLICA,
    "RISCO_ESTRUTURAL": RISCO_ESTRUTURAL,
    "EXPLOSIVOS_E_PRODUTOS_PERIOGOSOS": EXPLOSIVOS_E_PRODUTOS_PERIGOSOS,
    "PESSOAS_DESAPARECIDAS": PESSOAS_DESAPARECIDAS,
    "MATERIAIS_OBJETOS": MATERIAIS_OBJETOS,
}

In [22]:
def construir_mapa_reverso_natureza():
    natureza_para_categoria = {}
    duplicadas = {}

    for categoria, valores in MAPAS_NATUREZA_DESCRICAO.items():
        for valor in valores:
            if valor in natureza_para_categoria:
                duplicadas.setdefault(valor, set()).update(
                    {natureza_para_categoria[valor], categoria}
                )
            natureza_para_categoria[valor] = categoria 
    
    if duplicadas:
        raise ValueError(f"Naturezas em mais de uma categoria: {duplicadas}")
    
    return natureza_para_categoria

In [23]:
def obter_primeira_categoria(row, natureza_para_categoria):
    for valor in row: 
        if pd.isna(valor):
            continue 

        categoria = natureza_para_categoria.get(valor)

        if categoria is not None: 
            return categoria 
    
    return pd.NA

In [24]:
def aplicar_mapas_natureza_descricao(df, colunas=COLUNA_NATUREZA_DESCRICAO):
    natureza_cols = [col for col in colunas if col in df.columns]
    natureza_df = df[natureza_cols].copy()

    for col in natureza_cols:
        natureza_df[col] = natureza_df[col].apply(limpar_texto)
        natureza_df[col] = natureza_df[col].replace(CORRECOES)

    mask_sem_natureza = natureza_df.notna().any(axis=1)
    df = df.loc[mask_sem_natureza].copy()
    natureza_df = natureza_df.loc[mask_sem_natureza].copy()

    mask_tentativa = natureza_df.isin(INDETERMINADO).any(axis=1)
    df = df.loc[~mask_tentativa].copy()
    natureza_df = natureza_df.loc[~mask_tentativa].copy()

    natureza_para_categoria = construir_mapa_reverso_natureza()

    categoria_natureza = natureza_df.apply(
        obter_primeira_categoria,
        axis=1,
        natureza_para_categoria=natureza_para_categoria,
    )

    sem_categoria = categoria_natureza.isna()

    if sem_categoria.any():
        valores_sem_categoria = sorted(
            set(natureza_df.loc[sem_categoria].stack().dropna())
            - set(natureza_para_categoria)
            - set(INDETERMINADO)
        )

        raise ValueError(
            f"{sem_categoria.sum()} linhas ficaram sem categoria."
            f"Valores nao mapeados: {valores_sem_categoria}"
        )
    
    categorias = list(MAPAS_NATUREZA_DESCRICAO.keys())

    for categoria in categorias:
        df[categoria] = (categoria_natureza == categoria).astype("int64")
    
    soma_categorias = df[categorias].sum(axis=1)

    if not (soma_categorias == 1).all():
        raise ValueError(
            "As categorias de natureza nao ficaram boolenas exclusivas"
        )
    
    return df

## 07. Pipeline de limpeza de dados

Iremos executar um pipeline inicial de limpeza de dados executando as funções que construímos acima:

- `limpar_texto`;
- `mapear_binario`;
- `mapear_dia_semana`
- `tratar_coluna_hora`
- `adicionar_periodo_dia`

e realizar drop de colunas que não serão necessárias.

In [25]:
def executar_pipeline(arquivo):
    df = pd.read_csv(arquivo, sep=";", encoding="latin1", dtype=str)

    for col in COLUNAS_TEXTO:
        if col in df.columns:
            df[col] = df[col].apply(limpar_texto)
    
    if COLUNA_BAIRRO_NOME in df.columns:
        df[COLUNA_BAIRRO_NOME] = df[COLUNA_BAIRRO_NOME].apply(padronizar_bairro)
    
    df = df.dropna(subset=[COLUNA_BAIRRO_NOME])
        
    for col in COLUNAS_BINARIAS:
        if col in df.columns:
            df[col] = df[col].apply(mapear_binario).astype("Int64")

    for col in COLUNA_DIA_SEMANA:
        if col in df.columns:
            df[col] = df[col].apply(mapear_dia_semana).astype("Int64")

    df = tratar_coluna_hora(df, COLUNA_HORA)
    df = adicionar_periodo_dia(df, f"{COLUNA_HORA}_HORA")
    df = aplicar_mapas_natureza_descricao(df)

    colunas_para_remover = [col for col in DEFAULT_DROP_COLUMNS if col in df.columns]
    if colunas_para_remover:
        df = df.drop(columns=colunas_para_remover)
    
    caminho_saida = output_dir / arquivo.name 
    df.to_csv(caminho_saida, sep=";", index=False, encoding="utf-8")

    return df

In [26]:
df_limpo = executar_pipeline(arquivo)

In [27]:
df_limpo.head(10)

,ATENDIMENTO_BAIRRO_NOME,FLAG_EQUIPAMENTO_URBANO,FLAG_FLAGRANTE,LOGRADOURO_NOME,NATUREZA1_DEFESA_CIVIL,NATUREZA1_DESCRICAO,NATUREZA2_DEFESA_CIVIL,NATUREZA2_DESCRICAO,NATUREZA3_DEFESA_CIVIL,NATUREZA3_DESCRICAO,...,CRIME_ADMINISTRACAO_PUBLICA,CRIME_HONRA_DISCRIMINACAO,CRIME_CRIANCA_ADOLESCENTE,CRIME_FRAUDE_DOCUMENTAL,CRIME_DROGAS_SUBSTANCIAS,CRIME_ORDEM_PUBLICA,RISCO_ESTRUTURAL,EXPLOSIVOS_E_PRODUTOS_PERIOGOSOS,PESSOAS_DESAPARECIDAS,MATERIAIS_OBJETOS
0,cidade industrial,0,0,davi xavier da silva,0,alarmes,0,NaN,0,NaN,...,0,0,0,0,0,0,0,0,0,0
1,fazendinha,1,0,carlos klemtz,0,roubo,0,NaN,0,NaN,...,0,0,0,0,0,0,0,0,0,0
2,uberaba,0,0,doutor joao de paula moura brito,0,animais,0,NaN,0,NaN,...,0,0,0,0,0,0,0,0,0,0
3,sitio cercado,0,0,edgard cavalcanti de albuquerque,0,animais,0,NaN,0,NaN,...,0,0,0,0,0,0,0,0,0,0
4,tatuquara,1,0,carlos munhoz da rocha,0,alarmes,0,NaN,0,NaN,...,0,0,0,0,0,0,0,0,0,0
5,sitio cercado,0,0,hussein ibrahim omairy,0,alarmes,0,NaN,0,NaN,...,0,0,0,0,0,0,0,0,0,0
6,santa candida,1,0,nova de colombo,0,transito,0,NaN,0,NaN,...,0,0,0,0,0,0,0,0,0,0
7,boqueirao,1,0,carlos essenfelder,0,alarmes,0,NaN,0,NaN,...,0,0,0,0,0,0,0,0,0,0
8,centro,0,1,presidente faria,0,roubo,0,NaN,0,NaN,...,0,0,0,0,0,0,0,0,0,0
9,boa vista,1,0,santa edwiges,0,invasao,0,NaN,0,NaN,...,0,0,0,0,0,0,0,0,0,0


In [28]:
df_limpo.columns

Index(['ATENDIMENTO_BAIRRO_NOME', 'FLAG_EQUIPAMENTO_URBANO', 'FLAG_FLAGRANTE',
       'LOGRADOURO_NOME', 'NATUREZA1_DEFESA_CIVIL', 'NATUREZA1_DESCRICAO',
       'NATUREZA2_DEFESA_CIVIL', 'NATUREZA2_DESCRICAO',
       'NATUREZA3_DEFESA_CIVIL', 'NATUREZA3_DESCRICAO',
       'NATUREZA4_DEFESA_CIVIL', 'NATUREZA4_DESCRICAO',
       'NATUREZA5_DEFESA_CIVIL', 'NATUREZA5_DESCRICAO', 'OCORRENCIA_ANO',
       'OCORRENCIA_DIA_SEMANA', 'OCORRENCIA_MES', 'SECRETARIA_SIGLA',
       'SERVICO_NOME', 'NUMERO_PROTOCOLO_156', 'OCORRENCIA_DIA',
       'OCORRENCIA_HORA_HORA', 'OCORRENCIA_HORA_MINUTO', 'MADRUGADA', 'MANHA',
       'TARDE', 'NOITE', 'CRIME_VIOLENTO',
       'ATENDIMENTO_OPERACIONAL_ASSISTENCIAL', 'ACIDENTE_TRANSITO',
       'ACIDENTE_NATURAL', 'CRIME_PATRIMONIAL', 'CRIME_ADMINISTRACAO_PUBLICA',
       'CRIME_HONRA_DISCRIMINACAO', 'CRIME_CRIANCA_ADOLESCENTE',
       'CRIME_FRAUDE_DOCUMENTAL', 'CRIME_DROGAS_SUBSTANCIAS',
       'CRIME_ORDEM_PUBLICA', 'RISCO_ESTRUTURAL',
       'EXPLOSIVOS_E_PR